In [1]:
%load_ext autoreload
%autoreload 2

import os
import time
import numpy as np
from itertools import chain
import pickle

import sys
sys.path.append('/Users/tinnguyen/BlackBox/PointRegistration/source')

from cylinder_extraction.read_pts import read_input_data
# from ..cylinder_extraction import cells_classification, cells_connectivity
# from cell_grid import cell_grid, cell_grid_modify
from cylinder_extraction.write_output import write_array2txt
# from config import threshold
# from estimate_cylinder import shell_wall_extraction
# from voxel_grid import voxel_grid, voxelgridmodify
# from cylinder import Cylinder
# from utils import angle

In [18]:
data_dir = '../../data/External Scans/txt'
ptc_fn = 'Citgo Tank 4- Setup 006 - Cloud.e57'
ptc_col_names =  ["x","y","z"]

# read point cloud
_ = read_input_data(data_dir, ptc_fn)
src_pts_xyz = _[ptc_col_names].values

In [20]:
# 006 --> 004
translation = np.array(
    [[1.58724278],
    [5.81965091],
    [0.        ]]
)
rotation = np.array(
    [[ 0.75925727, -0.65079059,  0.        ],
    [ 0.65079059,  0.75925727,  0.        ],
    [ 0.,          0.,          1.        ]]
)

# 005 --> 004
# translation = np.array(
#     [[-2.98445259],
#     [ 6.0153626 ],
#     [ 0.        ]]
# )
# rotation = np.array(
#     [[-0.02210491, -0.99975566, 0.        ],
#     [ 0.99975566, -0.02210491,  0.        ],
#     [ 0.,          0.,          1.        ]]
# )

In [22]:
src_pts_xyz = src_pts_xyz.transpose()
translated_xyz = (rotation@src_pts_xyz) + translation

In [23]:
output_dir = os.path.join(data_dir, 'processing_outputs')
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

file_name = f"005_translated.txt"
file_path = os.path.join(output_dir, file_name)
write_array2txt(
    file_path, 
    translated_xyz.transpose()
)

In [18]:
src_pts_xyz = {}
data_dir = '../../data/External Scans/txt'
output_dir = os.path.join(data_dir, 'processing_outputs')
for setup in ['005', '006']:
    ptc_fn = f'{setup}_translated.txt'
    ptc_col_names =  ["x","y","z"]

    # read point cloud
    _ = read_input_data(output_dir, ptc_fn)
    src_pts_xyz[setup] = _[ptc_col_names].values
data = np.concatenate(list(src_pts_xyz.values()))

In [20]:
ptc_fn = 'Citgo Tank 4- Setup 004 - Cloud.e57'
ptc_col_names =  ["x","y","z"]

# read point cloud
_ = read_input_data(data_dir, ptc_fn)
src_pts_xyz = _[ptc_col_names].values

In [22]:
data = np.concatenate([data, src_pts_xyz])

In [23]:
file_name = f"cloud_full_translated.txt"
file_path = os.path.join(output_dir, file_name)
write_array2txt(
    file_path,
    data
)

In [25]:
import pandas as pd

In [42]:
group_df = pd.read_csv('cylinder_group.txt', delimiter='\t')
group_df.reset_index(inplace=True)
group_df

,index,setup,tank_id group
0,4,1,0
1,4,3,1
2,4,4,2
3,4,5,3
4,4,8,4
5,4,10,5
6,5,4,3
7,5,5,2
8,5,7,1
9,6,6,0


In [43]:
group_df.columns = ['setup', 'tank_id', 'group']
group_df

,setup,tank_id,group
0,4,1,0
1,4,3,1
2,4,4,2
3,4,5,3
4,4,8,4
5,4,10,5
6,5,4,3
7,5,5,2
8,5,7,1
9,6,6,0


In [45]:
output_dir

'../../data/External Scans/txt/processing_outputs'

In [47]:
groupby_df = group_df.groupby('group')

In [ ]:
def rename_file(row):
    old_fn = f'Citgo Tank 4- Setup 00{row.setup} - Cloud.e57 cluster_{row.tank_id}.txt'
    new_fn = f'tank {row.group} - Setup 00{row.setup}.{tank_id}'
    os.rename(old_file, new_file)

In [66]:
# 006 --> 004
transformation = {
    6: {
        'translation': np.array(
            [[1.58724278],
            [5.81965091],
            [0.        ]]
        ),
        'rotation': np.array(
            [[ 0.75925727, -0.65079059,  0.        ],
            [ 0.65079059,  0.75925727,  0.        ],
            [ 0.,          0.,          1.        ]]
        )
    },
    5: {
        'translation': np.array(
            [[-2.98445259],
            [ 6.0153626 ],
            [ 0.        ]]
        ),
        'rotation': np.array(
            [[-0.02210491, -0.99975566, 0.        ],
            [ 0.99975566, -0.02210491,  0.        ],
            [ 0.,          0.,          1.        ]]
        )
    }
}
# translation = np.array(
#     [[1.58724278],
#     [5.81965091],
#     [0.        ]]
# )
# rotation = np.array(
#     [[ 0.75925727, -0.65079059,  0.        ],
#     [ 0.65079059,  0.75925727,  0.        ],
#     [ 0.,          0.,          1.        ]]
# )

# 005 --> 004
# translation = np.array(
#     [[-2.98445259],
#     [ 6.0153626 ],
#     [ 0.        ]]
# )
# rotation = np.array(
#     [[-0.02210491, -0.99975566, 0.        ],
#     [ 0.99975566, -0.02210491,  0.        ],
#     [ 0.,          0.,          1.        ]]
# )

In [70]:
for gr in groupby_df:
    gr_df = gr[1]
    for i in range(gr_df.shape[0]):
        row = gr_df.iloc[i]
        old_fn = f'Citgo Tank 4- Setup 00{row.setup} - Cloud.e57 cluster_{row.tank_id}.txt'
        old_fp = os.path.join(output_dir, old_fn)
        new_fn = f'tank {row.group} - Setup 00{row.setup}.{row.tank_id}.txt'
        new_fp = os.path.join(output_dir, new_fn)

        _ = read_input_data(output_dir, old_fn)
        src_pts_xyz = _[ptc_col_names].values
        src_pts_xyz = src_pts_xyz.transpose()

        if row.setup == 4:
            translated_xyz = src_pts_xyz.transpose()
        else:
            rotation = transformation[row.setup]['rotation']
            translation = transformation[row.setup]['translation']
            translated_xyz = (rotation@src_pts_xyz) + translation
            translated_xyz = translated_xyz.transpose()

        write_array2txt(
            new_fp, 
            translated_xyz
        )

In [57]:
row

setup      4
tank_id    1
group      0
Name: 0, dtype: int64